In [1]:
#bash
#!pip install -q flwr[simulation] flwr_datasets[vision] torch torchvision matplotlib tensorflow scikit-learn 

#zsh
#!pip install -q flwr\[simulation\] flwr_datasets\[vision\] torch torchvision matplotlib tensorflow scikit-learn 

In [2]:
# Import necessary libraries
import flwr as fl
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import logging
import random
import ray

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix
from tensorflow import keras
from tensorflow.keras import layers
from datasets.utils.logging import disable_progress_bar
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Set up logging
logging.basicConfig(level=logging.INFO)

DEVICE = torch.device("cpu")  # Try "cuda" to train on GPU
#DEVICE = torch.device("cuda")  # Try "cuda" to train on GPU
print(
    f"Training on {DEVICE} using PyTorch {torch.__version__} and Flower {fl.__version__}"
)
disable_progress_bar()

c:\Users\tulio\repos\Mestrado\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-09-13 14:08:34,811	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Training on cpu using PyTorch 2.4.1+cpu and Flower 1.11.1


In [3]:
# Define the path to the dataset and the sequence length
#PATH = '/Users/tuliopolido/DBs/vivabem_aerobic_65users.csv'
#PATH = 'D:/DBs/vivabem_aerobic_65users.csv'
PATH = '/home/tulionpl/DBs/vivabem_aerobic_65users_2.csv'
SEQUENCE_LENGTH = 7
NUM_CLASSES = 3
BATCH_SIZE = 16
NUM_EPOCHS = 5
LEARNING_RATE = 0.005 
NUM_CLIENTS = 20

# Define client's resources
CLIENT_RESOURCES = {'num_cpus': 2.0}

ID_COLUMN = 'int_id'

In [4]:
def load_wisdm_data(file_path, user_percentage=100):
    data = []
    labels = []
    users = []
    
    with open(file_path) as file:
        for line in file:
            line = line.strip().strip(';').split(',')
            if len(line) == 6:
                user = line[0]
                activity = line[1]
                timestamp = line[2]
                try:
                    x = float(line[3])
                except ValueError:
                    x = 0.0
                try:
                    y = float(line[4])
                except ValueError:
                    y = 0.0
                try:
                    z = float(line[5])
                except ValueError:
                    z = 0.0
                
                data.append([x, y, z])
                labels.append(activity)
                users.append(user)
    
    # Convert to numpy arrays
    data = np.array(data)
    labels = np.array(labels)
    users = np.array(users)
    
    # Filter users
    unique_users = np.unique(users)
    num_users = len(unique_users)
    print(num_users)
    num_selected_users = int(user_percentage / 100 * num_users)
    print(num_selected_users)
    selected_users = set(random.sample(list(unique_users), num_selected_users))
    
    selected_indices = [i for i, user in enumerate(users) if user in selected_users]
    
    filtered_data = data[selected_indices]
    filtered_labels = labels[selected_indices]
    
    return filtered_data, filtered_labels

In [30]:
file_path = '../datasets/WISDM_ar_v1.1_raw.txt'
data, labels = load_wisdm_data(file_path, user_percentage=80)
print(f'Data shape: {data.shape}, Labels shape: {labels.shape}')

36
28
Data shape: (834422, 3), Labels shape: (834422,)


In [31]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Encode labels to integers
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)

# Standardize the data
scaler = StandardScaler()
data = scaler.fit_transform(data)

# Reshape data into sequences
def create_sequences(data, labels, sequence_length):
    sequences = []
    seq_labels = []
    for i in range(len(data) - sequence_length):
        seq = data[i:i+sequence_length]
        label = labels[i+sequence_length-1]
        sequences.append(seq)
        seq_labels.append(label)
    
    return np.array(sequences), np.array(seq_labels)

data, labels = create_sequences(data, labels, SEQUENCE_LENGTH)

# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)
x_train = np.transpose(x_train, (0, 2, 1))
x_test = np.transpose(x_test, (0, 2, 1))


# Convert data to PyTorch tensors
train_dataset = TensorDataset(torch.tensor(x_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(torch.tensor(x_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f'Training data shape: {x_train.shape}, Training labels shape: {y_train.shape}')
print(f'Testing data shape: {x_test.shape}, Testing labels shape: {y_test.shape}')

Training data shape: (667532, 3, 7), Training labels shape: (667532,)
Testing data shape: (166883, 3, 7), Testing labels shape: (166883,)


In [32]:
class PEN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(PEN, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv1d(in_channels=input_size, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2, padding=0)

        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        
        # LSTM layers
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=2, batch_first=True)
        
        # Fully connected layers
        self.fc = nn.Linear(hidden_size + 128, num_classes)

    def forward(self, x):
        # Convolutional path
        x_cnn = self.conv1(x)
        x_cnn = self.bn1(x_cnn)
        x_cnn = self.relu(x_cnn)
        x_cnn = self.pool(x_cnn)

        x_cnn = self.conv2(x_cnn)
        x_cnn = self.bn2(x_cnn)
        x_cnn = self.relu(x_cnn)
        x_cnn = self.pool(x_cnn)
        
        x_cnn = x_cnn.mean(dim=-1)  # Global Average Pooling

        # LSTM path
        x_lstm, _ = self.lstm(x.transpose(1, 2))  # Transpose to match LSTM input shape
        x_lstm = x_lstm[:, -1, :]  # Take the last output of the sequence

        # Concatenate CNN and LSTM paths
        x = torch.cat((x_cnn, x_lstm), dim=1)

        # Fully connected layer
        x = self.fc(x)
        return x

In [33]:
# Training function
def train(model, train_loader, criterion, optimizer, num_epochs):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

In [34]:
# Testing function
def test(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            outputs = model(x_batch)
            _, predicted = torch.max(outputs.data, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    accuracy = 100 * correct / total
    print(f'Accuracy: {accuracy:.2f}%')


In [35]:
# Define model, criterion, and optimizer
input_size = x_train.shape[1]
hidden_size = 128
num_classes = len(np.unique(y_train))

model = PEN(input_size, hidden_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [36]:
# Train the model
num_epochs = 10
train(model, train_loader, criterion, optimizer, num_epochs)

# Test the model
test(model, test_loader)

Epoch [1/10], Loss: 0.3792
Epoch [2/10], Loss: 0.2163
Epoch [3/10], Loss: 0.1689
Epoch [4/10], Loss: 0.1409
Epoch [5/10], Loss: 0.1202
Epoch [6/10], Loss: 0.1050
Epoch [7/10], Loss: 0.0928
Epoch [8/10], Loss: 0.0835
Epoch [9/10], Loss: 0.0755
Epoch [10/10], Loss: 0.0697
Accuracy: 95.31%
